# TTFT breakdown & CDF — 検証2-1-2(KVキャッシュ考慮, N=50, 1:2負荷, RTX4090)

今回のテストで読み込むCSV:
- `results/exp212-1to2-nearest-kv.csv`
- `results/exp212-1to2-nearest-migrate.csv`
- `results/exp212-1to2-nearest-migrate-kv.csv`

出力画像:
- `outputs/image/2026-07-05_exp212_n50_1to2_rtx4090_kv_ttft_breakdown.png`
- `outputs/image/2026-07-05_exp212_n50_1to2_rtx4090_kv_ttft_cdf.png`


In [ ]:
import csv, math, statistics as st
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(6):
        if (cur / "serving").exists() and (cur / "results").exists():
            return cur
        cur = cur.parent
    raise RuntimeError("Could not locate repo root")

REPO_ROOT = find_repo_root(Path.cwd())
CSV_PATHS = {
    "Method A: NEAREST_KV": REPO_ROOT / "results/exp212-1to2-nearest-kv.csv",
    "Method B: NEAREST_MIGRATE": REPO_ROOT / "results/exp212-1to2-nearest-migrate.csv",
    "Method C: NEAREST_MIGRATE_KV": REPO_ROOT / "results/exp212-1to2-nearest-migrate-kv.csv",
}
OUT_DIR = REPO_ROOT / "outputs/image"
PREFIX = "2026-07-05_exp212_n50_1to2_rtx4090_kv_ttft"


In [ ]:
def load_rows(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))

def fnum(row, key):
    v = row.get(key, "")
    return float(v) if v not in ("", None) else 0.0

def vals_ms(rows, key):
    return [fnum(r, key) / 1e6 for r in rows]

def pct(vals, q):
    vals = sorted(vals)
    pos = (len(vals) - 1) * q
    lo = math.floor(pos); hi = math.ceil(pos)
    if lo == hi:
        return vals[lo]
    return vals[lo] * (hi - pos) + vals[hi] * (pos - lo)

def breakdown(rows):
    kv = st.mean(vals_ms(rows, "kv_migration_latency_ns"))
    comm = st.mean(vals_ms(rows, "communication_latency_ns"))
    return {
        "Queue": st.mean(vals_ms(rows, "queueing_before_ttft_ns")),
        "KV transfer": kv,
        "Compute": st.mean(vals_ms(rows, "prefill_service_ns")),
        "RTT/comm": max(0, comm - kv),
    }

rows_by = {label: load_rows(path) for label, path in CSV_PATHS.items()}
for label, rows in rows_by.items():
    print(label, len(rows), breakdown(rows))


## 1. 平均TTFTの内訳


In [ ]:
from IPython.display import Image
Image(filename=str(OUT_DIR / "2026-07-05_exp212_n50_1to2_rtx4090_kv_ttft_breakdown.png"))


## 2. TTFT CDF


In [ ]:
from IPython.display import Image
Image(filename=str(OUT_DIR / "2026-07-05_exp212_n50_1to2_rtx4090_kv_ttft_cdf.png"))
